In [1]:
!pip install -q pymupdf sentence-transformers groq gradio numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 10.5 MB/s eta 0:00:00


In [2]:
import os, getpass
from groq import Groq

key = getpass.getpass("Paste your Groq API key: ").strip()
print("length:", len(key), "| starts with:", key[:4])
groq_client = Groq(api_key=key)

available = [m.id for m in groq_client.models.list().data]
preferred = ["llama-3.3-70b-versatile", "openai/gpt-oss-120b", "openai/gpt-oss-20b",
             "llama-3.1-8b-instant", "qwen/qwen3-32b"]
MODEL = next((m for m in preferred if m in available), None)
if MODEL is None:
    MODEL = next(m for m in available if not any(x in m for x in ("whisper", "guard", "tts", "orpheus")))
print("Using model:", MODEL)

Paste your Groq API key: ··········
length: 56 | starts with: gsk_
Using model: openai/gpt-oss-120b


In [3]:
import os, pymupdf, numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
chunks = []
vecs = np.zeros((0, 384), dtype="float32")

SYSTEM = (
    "You are an electrical engineering study assistant. Answer using ONLY the numbered "
    "context passages provided. Cite sources like [1] or [2] after the claims they support. "
    "Keep formulas, values and units exactly as written in the context; never invent numbers. "
    "If the context doesn't contain the answer, say you couldn't find it in the uploaded documents. "
    "Explain step by step for a student. For anything safety-critical (wiring, high voltage, "
    "code compliance), remind the user to follow the local electrical code and consult a "
    "licensed electrician."
)

def add_pdfs(files):
    global chunks, vecs
    if not files:
        return "Please upload at least one PDF first."
    chunks, vecs = [], np.zeros((0, 384), dtype="float32")   # fresh index each time
    for f in files:
        path = f if isinstance(f, str) else f.name
        name = os.path.basename(path)
        new = []
        for page_no, page in enumerate(pymupdf.open(path), 1):
            words = page.get_text().split()
            if len(words) < 30:
                continue
            for start in range(0, len(words), 300):
                new.append({"source": name, "page": page_no,
                            "text": " ".join(words[start:start + 350])})
                if start + 350 >= len(words):
                    break
        if new:
            v = embedder.encode([c["text"] for c in new], batch_size=64,
                                normalize_embeddings=True)
            chunks += new
            vecs = np.vstack([vecs, v])
    return f"Done! {len(chunks)} chunks indexed. You can ask questions now."

def answer(question):
    if not chunks:
        return "Upload and index a PDF first.", ""
    if not question.strip():
        return "Type a question first.", ""
    q = embedder.encode(["Represent this sentence for searching relevant passages: " + question],
                        normalize_embeddings=True)[0]
    top = np.argsort(-(vecs @ q))[:5]
    hits = [chunks[i] for i in top]
    context = "\n\n".join(f"[{n}] ({h['source']}, page {h['page']})\n{h['text']}"
                          for n, h in enumerate(hits, 1))
    reply = groq_client.chat.completions.create(
        model=MODEL, max_tokens=1500,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}])
    sources = "\n\n".join(f"**[{n}] {h['source']}, page {h['page']}**: {h['text'][:250]}..."
                          for n, h in enumerate(hits, 1))
    return reply.choices[0].message.content, sources

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import gradio as gr

with gr.Blocks(title="Electrical RAG") as demo:
    gr.Markdown("# ⚡ Electrical RAG\nUpload your PDFs, click **Index documents**, then ask questions.")
    files = gr.File(label="Upload PDFs", file_count="multiple", file_types=[".pdf"])
    index_btn = gr.Button("Index documents")
    status = gr.Textbox(label="Status", interactive=False)
    question = gr.Textbox(label="Your question", placeholder="e.g. State Kirchhoff's voltage law")
    ask_btn = gr.Button("Ask", variant="primary")
    answer_box = gr.Markdown()
    with gr.Accordion("Sources", open=False):
        sources_box = gr.Markdown()

    index_btn.click(add_pdfs, files, status)
    ask_btn.click(answer, question, [answer_box, sources_box])
    question.submit(answer, question, [answer_box, sources_box])

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5794977b9705904cf3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
